# Goals

Develop explanatory GEP x ligand matrix for use in ridge regression models of ligand activity prediction. We want to:

* Remove ligands that have weak or no effects on sPCA component scores
* Aggregate highly co-linear signaling activities

# Import

In [1]:
import scanpy as sc
import pandas as pd
import decoupler as dc
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

import os
import sys
# Repo-relative paths (inputs: imports_stable/, outputs: analysis_outs/)
from pathlib import Path
_here = Path.cwd().resolve()
REPO_DIR = str(next(p for p in [_here, *_here.parents] if (p / "imports_stable").is_dir()))
IMPORTS_DIR = f"{REPO_DIR}/imports_stable"
OUTS_DIR = f"{REPO_DIR}/analysis_outs/03_activity_inference_model"
os.makedirs(f"{OUTS_DIR}/inference_model_final", exist_ok=True)
sys.path.insert(0, f"{REPO_DIR}/functions")
import scanpy_custom as scc

%load_ext autoreload
%autoreload 2

In [2]:
# import sPCA components and LM results
spca_components = pd.read_csv(IMPORTS_DIR + "/SIG13/analysis_outs/spca/zscore_degs_allLigands_0.1_alpha1.0_sPCA_loadings.csv")
lm_scored = pd.read_csv(IMPORTS_DIR + "/SIG13/analysis_outs/spca/lm_scored_zscore_degs_allLigands_0.1_alpha1.0_sPCA_clean.csv")
lm_fit = pd.read_csv(IMPORTS_DIR + "/SIG13/analysis_outs/spca/lm_fit_zscore_degs_allLigands_0.1_alpha1.0_sPCA_clean.csv")

# import modal hdbscan clusters (bootstrap stability analysis, see analysis/02_combinatorial_screen_signalseq_SIG13/13_clustering)
hdbscan_clusters = pd.read_csv(IMPORTS_DIR + "/SIG13/analysis_outs/clustering/hdbscan_bootstrap_modal_clusters_minSamples1.csv")
hdbscan_clusters_single = pd.read_csv(IMPORTS_DIR + "/SIG13/analysis_outs/clustering/hdbscan_bootstrap_modal_clusters_singleLigand_minSamples1.csv")

# import inter-replicate correlation based on interaction LFCs and total LFCs
interRepCor = pd.read_csv(IMPORTS_DIR + "/SIG13/analysis_outs/replicate_corr/replicate_correlation_interactionLfc_0.2filter.csv")
totalRepCor = pd.read_csv(IMPORTS_DIR + "/SIG13/analysis_outs/replicate_corr/replicate_correlation_singleLfc_0.2filter.csv")

In [3]:
# import SIG13 anndata
rna = sc.read_h5ad(f"{IMPORTS_DIR}/SIG13/scanpy_outs/SIG13_doublets_DSB7.h5ad")

# SIG13 Scoring

First, we will calculate sPCA scores for the components using a weighted mean of scaled log1p_norm component genes. Then, I will calculate the difference between stimulation conditions and linker_linker to create the explanatory matrix.

In [4]:
# filter for good components (those not strongly associated with replicates)
good_comps = lm_scored['component'].unique().tolist()
spca_components = spca_components[spca_components['spca_component'].isin(good_comps)]
# format for decoupler
net = spca_components.rename(columns={'gene':'target',
                                      'spca_component':'source',
                                      'loading':'weight'})
# define genes used in scoring
comp_genes = net['target'].unique().tolist()

In [5]:
# subset and scale 
rna_sub = rna[:,comp_genes].copy()
rna_sub.X = rna_sub.layers['log1p_norm']
sc.pp.scale(rna_sub)

# calculate pseudobulk 
rna_pb = sc.get.aggregate(rna_sub, by=['ligand_call_DSB7'], func='mean')

/data1/rudenska/EYW/mamba_envs/scanpy_standard2/lib/python3.14/functools.py:982: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


In [6]:
# calculate waggr without any permutations (no pvalues needed)
dc.mt.waggr(rna_pb, net, tmin=5, verbose=True, layer='mean', times=0)

# Extract score_waggr and padj_waggr from rna_pb.obsm
score_df = pd.DataFrame(rna_pb.obsm['score_waggr'], 
                        index=rna_pb.obs_names, 
                        columns=rna_pb.obsm['score_waggr'].columns)

# Calculate differential scores by subtracting linker_linker scores from all conditions
linker_scores = score_df.loc['linker_linker']
score_df = score_df.subtract(linker_scores, axis=1)

2026-07-09 21:51:25 | [INFO] waggr - Running waggr
2026-07-09 21:51:25 | [INFO] Extracted omics mat with 660 rows (observations) and 5246 columns (features)
2026-07-09 21:51:25 | [INFO] Network adjacency matrix has 5246 unique features and 68 unique sources
2026-07-09 21:51:25 | [INFO] waggr - using function _wmean
2026-07-09 21:51:25 | [INFO] waggr - using _wmean for the first time, will need to be compiled
2026-07-09 21:51:25 | [INFO] waggr - calculating scores for 68 sources across 660 observations
2026-07-09 21:51:26 | [INFO] waggr - adjusting p-values by FDR
2026-07-09 21:51:26 | [INFO] waggr - done


# Activity Selection

Next, we filter for ligands that have a significant effect on sPCA components and are well correlated between replicates. We will also aggregate similar activities into consensus activities.

## Select Single Activities

In [7]:
# select single ligands based on pearson correlation cutoff
consistent_ligands = totalRepCor.loc[((totalRepCor['pearson_corr'] > 0.25))]
consistent_ligands.loc[:,'condition'] = consistent_ligands['condition'].str.replace('linker_|_linker','', regex=True)
consistent_ligands = consistent_ligands['condition'].unique()

# pull out single terms from lm of spca component scores
spca_lm_singles = lm_fit[~lm_fit.term.str.contains(':')]
spca_lm_singles.loc[:,'term'] = spca_lm_singles['term'].str.replace('ligand1|ligand2','', regex=True)

# define active ligands
spca_active_ligands = spca_lm_singles.query('p_adj < 0.01 & term in @consistent_ligands')['term'].unique()

In [8]:
spca_active_ligands

array(['IFNG', 'IL2', 'IL21', 'IL27', 'IL4', 'IL6', 'IFNA', 'IFNB1',
       'IFNK', 'TNFSF18', 'IL7', 'TNF', 'TNFSF15', 'TGFB1', 'TNFSF4',
       'TSLP', 'TNFSF9', 'CCL21A', 'CCL19'], dtype=object)

## Aggregate Single Activities

To aggregate single ligand activities, we use the modal HDBSCAN clusters from the bootstrap stability analysis (`analysis/02_combinatorial_screen_signalseq_SIG13/13_clustering`) to group co-linear single-ligand conditions into consensus activities.

In [37]:
# join active single-ligand conditions with the modal HDBSCAN clusters
# (bootstrap-stability clustering; -1 = not assigned to a stable cluster)
score_df_linker = (score_df
                   .query("index.str.contains('linker')", engine='python')
                   .copy())
score_df_linker['ligand'] = score_df_linker.index.str.replace('linker_|_linker', '', regex=True)
score_df_linker_filtered = (score_df_linker
 .query("ligand.isin(@spca_active_ligands)", engine='python')
 .drop(columns=['ligand'])
 .merge(hdbscan_clusters_single[['condition','cluster']],
        left_index=True, right_on='condition')
 .set_index('condition'))

# Determine the "Name" for each cluster
# (We pick the ligand condition with the highest row mean to represent the cluster)
cluster_names = (score_df_linker_filtered
                 .query('cluster != -1')
                 .assign(row_mean=lambda x: x.drop(columns=['cluster']).mean(axis=1))
                 .groupby('cluster')['row_mean']
                 .idxmax()) # Returns a Series mapping Cluster ID -> Ligand Condition
cluster_names = cluster_names + '_c'

# make df of activity names
single_names = score_df_linker_filtered.reset_index()
single_names = single_names[['condition', 'cluster']]
single_names = single_names.merge(cluster_names, left_on='cluster', right_index=True, how='left')
single_names['row_mean'] = single_names['row_mean'].fillna(single_names['condition'])

# Calculate the Mean of the groups and rename the index
score_df_linker_grouped = (score_df_linker_filtered
                            .query('cluster != -1')
                                 .groupby('cluster')
                                 .mean()
                                 # Map the numbered cluster index to the Representative Ligand Condition
                                 .set_axis(cluster_names, axis='index') 
                                 )

score_df_linker_unclustered = (score_df_linker_filtered
                                     .query('cluster == -1')
                                     .drop(columns=['cluster']))

# Combine clustered representatives and unclustered conditions
score_df_linker_agg = pd.concat([score_df_linker_grouped, score_df_linker_unclustered])

# remove bad TGFB1
score_df_linker_agg = score_df_linker_agg.drop(index='linker_TGFB1', errors='ignore')

# remove linker text in name
score_df_linker_agg.index = score_df_linker_agg.index.str.replace('linker_|_linker','', regex=True)

In [38]:
# get list of aggregated activities
score_df_linker_agg.index

Index(['CCL19_c', 'TNFSF18_c', 'IL21_c', 'IL7_c', 'IFNB1_c', 'IL2_c', 'IL4_c',
       'IFNG_c', 'IL27_c', 'TGFB1', 'TSLP'],
      dtype='object')

## Select Combinatorial Activities

In [39]:
# select consistent interactions based on pearson correlation
consistent_interactions = interRepCor.loc[((interRepCor['pearson_corr'] > 0.25)), 
                                     'interaction'].unique()

# filter for interactions with significant interaction effects in component scores and good correlation
spca_active_interactions = (lm_scored
                            .query('p_adj_interaction < 0.001 & interaction in @consistent_interactions')
                            )['interaction'].unique()

In [40]:
len(spca_active_interactions)

133

In [41]:
spca_active_interactions

array(['IL2_IL2', 'IL2_CCL21A', 'IL2_IFNA', 'IL2_IFNB1', 'IL2_IFNK',
       'IL2_IL21', 'IL2_IL27', 'IL2_IL4', 'IL2_IL6', 'IL2_IL7', 'IL2_TNF',
       'IL2_TNFSF15', 'IL2_TNFSF18', 'IL2_TNFSF4', 'IL21_IL2',
       'IL21_IL21', 'IL21_IL4', 'IL27_IL2', 'IL27_IL7', 'IL4_CCL21A',
       'IL4_IFNA', 'IL4_IFNB1', 'IL4_IL2', 'IL4_IL21', 'IL4_IL4',
       'IL4_IL6', 'IL4_IL7', 'IL6_IL2', 'IL6_IL6', 'IL7_IFNA',
       'IL7_IFNB1', 'IL7_IL2', 'IL7_IL27', 'IL7_IL4', 'IL7_IL6',
       'TNF_IL2', 'IFNG_IFNB1', 'IL21_IL6', 'IL27_IL27', 'IL27_IL6',
       'IL6_IL27', 'TGFB1_CCL21A', 'TGFB1_IL21', 'TGFB1_IL4', 'TGFB1_IL6',
       'TNF_TNF', 'IFNG_IL7', 'IL21_IL7', 'IL21_TNFSF15', 'IL27_IL4',
       'IL4_IFNG', 'IL4_IL27', 'IL4_TNF', 'IL4_TNFSF18', 'IL6_IL4',
       'IL6_IL7', 'IL7_CCL21A', 'IL7_IFNG', 'IL7_IL21', 'IL7_IL7',
       'IL7_TNFSF15', 'TNF_TNFSF18', 'IL2_TNFSF9', 'IL21_TNF',
       'IL21_TNFSF18', 'IL4_TNFSF15', 'IL4_TNFSF4', 'IL6_TNFSF18',
       'IL7_TNF', 'IL7_TNFSF18', 'IL7_TNFSF4', 'TN

## Aggregate Combinatorial Activities

In [42]:
# isolate interaction conditions
score_df_interactions = (score_df
                         .query("index.str.contains('linker')==False", engine='python'))
# filter for selected combinations
score_df_interactions = score_df_interactions.loc[spca_active_interactions]
# join with modal hdbscan clusters (bootstrap stability analysis)
score_df_interactions = (score_df_interactions
                         .merge(hdbscan_clusters[['interaction','cluster']],
                                left_index=True, right_on='interaction')
                         .set_index('interaction'))

In [43]:
# list unclustered interactions
score_df_interactions.query('cluster == -1')['cluster']

interaction
IL2_CCL21A     -1
IL27_IL7       -1
IL7_IFNA       -1
IL7_IFNB1      -1
IL7_IL27       -1
IL7_IL6        -1
IFNG_IL7       -1
IL21_IL7       -1
IL4_IFNG       -1
IL6_IL7        -1
IL7_CCL21A     -1
IL7_IFNG       -1
IL7_IL21       -1
IL2_TNFSF9     -1
TNF_TSLP       -1
IFNG_IL21      -1
IFNG_TNFSF18   -1
IL21_IFNG      -1
IL27_TNFSF15   -1
IL6_IFNA       -1
IL6_IFNB1      -1
IL27_TNFSF18   -1
IL27_TNFSF4    -1
IFNG_TNF       -1
IFNG_IL4       -1
IFNG_CCL19     -1
IFNG_CCL21A    -1
IL2_TSLP       -1
IL6_IFNK       -1
TNF_TNFSF9     -1
IL6_CCL19      -1
IL6_CCL21A     -1
Name: cluster, dtype: int64

In [44]:
# remove clusters that represent double dose sitations
# 14: TNF+TNF, TNF+TNFSF18
# 26: IL2/4 + IL2/4
# 28: IFNG/IL27 + IFNG/IL27
# 30: IL21/6 + IL21/6
double_dose_hdbscan_clusters_1 = [14, 26, 30, 28]

In [45]:
# Define interactions that need to be aggregated
df_filtered = (score_df_interactions
               .query('cluster != -1')
               .query('cluster not in @double_dose_hdbscan_clusters_1'))

# Determine the "Name" for each cluster 
# (We pick the ligand name with the highest row mean to represent the cluster)
cluster_names = (df_filtered
                 .assign(row_mean=lambda x: x.drop(columns=['cluster']).mean(axis=1))
                 .groupby('cluster')['row_mean']
                 .idxmax()) # Returns a Series mapping Cluster ID -> Ligand Name
cluster_names = cluster_names + '_c'

# Calculate the Mean of the groups and rename the index
score_df_interactions_grouped = (df_filtered
                                 .groupby('cluster')
                                 .mean()
                                 # Map the numbered cluster index to the Representative Ligand Name
                                 .set_axis(cluster_names, axis='index') 
                                 )

# Combine clustered representatives and unclustered interactions
score_df_interactions_agg = pd.concat([score_df_interactions_grouped])

In [46]:
len(score_df_interactions_agg)

27

# Construct Final Explanatory Matrix

In [47]:
# export ligand conditions and associated consensus activity names
# get cluster identities
clustered_interaction_names = df_filtered[['cluster']].reset_index()
clustered_interaction_names = clustered_interaction_names.merge(cluster_names, left_on='cluster', right_index=True, how='left')
clustered_interaction_names['activity_type'] = "combinatorial"
single_names['activity_type'] = "single"

# rename  columns
clustered_interaction_names.columns = ['activity', 'group','annotation','activity_type']
single_names.columns = ['activity', 'group','annotation','activity_type']

# merge all by row
full_clusters = pd.concat([clustered_interaction_names, single_names])

In [48]:
# merge single and combinatorial activities
score_merge = pd.concat([score_df_interactions_agg, score_df_linker_agg], axis=0)

# remove components that are associated with replicate effects
score_filtered = score_merge[good_comps].copy()
score_filtered.index.name = 'interaction'

# shape of final explanatory matrix
score_filtered.shape

(38, 68)

In [49]:
# Export
score_filtered_long = score_filtered.reset_index().melt(id_vars='interaction', 
                                                      var_name='component', 
                                                      value_name='diff_score')
score_filtered.to_csv(f"{OUTS_DIR}/inference_model_final/SIG13_waggr_scores_explanatory_mat.csv")
score_filtered_long.to_csv(f"{OUTS_DIR}/inference_model_final/SIG13_waggr_scores_explanatory_long.csv", index=False)
full_clusters.to_csv(f"{OUTS_DIR}/inference_model_final/SIG13_waggr_activity_clusters.csv", index=False)